In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv("/content/IMDB Dataset.csv")

In [ ]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
data["sentiment"] =  data.sentiment.apply(lambda x: 1 if x == "positive" else 0)

In [ ]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lem = WordNetLemmatizer()
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
def text_preprocessing(text):
  text = text.lower()
  # Remove Punctuation
  text = "".join(i for i in text if i not in string.punctuation)
  # Remove Spaces
  words = text.split()
  # Remove Stopwords
  text = " ".join(word for word in words if word not in stopwords.words("english"))
  text = " ".join(lem.lemmatize(word) for word in text.split())
  return text

In [ ]:
data["review"].apply(text_preprocessing)

,review
0,one reviewer mentioned watching 1 oz episode y...
1,wonderful little production br br filming tech...
2,thought wonderful way spend time hot summer we...
3,basically there family little boy jake think t...
4,petter matteis love time money visually stunni...
...,...
49995,thought movie right good job wasnt creative or...
49996,bad plot bad dialogue bad acting idiotic direc...
49997,catholic taught parochial elementary school nu...
49998,im going disagree previous comment side maltin...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer = Tokenizer(num_words = 10000)

tokenizer.fit_on_texts(data["review"])

sequence = tokenizer.texts_to_sequences(data["review"])

padded_reviews= pad_sequences(sequence, maxlen = 500 , padding = "post")


In [ ]:
print(padded_reviews.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# 1. Model Architecture Define Karein
model = Sequential()

# Embedding Layer (Words ko dense vectors mein convert karegi)
model.add(Embedding(input_dim=10000, output_dim=64, input_length=500))

# RNN Layer (LSTM use kar rahe hain taaki sequence context achhe se yaad rahe)
model.add(LSTM(64, return_sequences=False))

# Dropout for regularization (overfitting rokne ke liye)
model.add(Dropout(0.5))

# Output Layer (Binary classification ke liye sigmoid activation)
model.add(Dense(1, activation='sigmoid'))

# 2. Model Compile Karein
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# 3. Model Train Karein
history = model.fit(
    padded_reviews,
    data["sentiment"].values,
    epochs=10,           # Aap epochs adjust kar sakte hain
    batch_size=64,
    validation_split=0.2 # 20% data validation ke liye
)

# 4. Evaluation (Optional: Agar test set ho)
# loss, accuracy = model.evaluate(X_test, y_test)
# print(f"Test Accuracy: {accuracy * 100:.2f}%")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.5087 - loss: 0.6931 - val_accuracy: 0.5126 - val_loss: 0.6930
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5095 - loss: 0.6885 - val_accuracy: 0.5140 - val_loss: 0.6873
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.5246 - loss: 0.6726 - val_accuracy: 0.5226 - val_loss: 0.6864
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5311 - loss: 0.6581 - val_accuracy: 0.5134 - val_loss: 0.6888
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5355 - loss: 0.6525 - val_accuracy: 0.5167 - val_loss: 0.7029
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5342 - loss: 0.6472 - val_accuracy: 0.5147 - val_loss: 0.7152
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5379 - loss: 0.6476 - val_accuracy: 0.5137 - val_loss: 0.7286
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.5396 - loss: 0.6451 - 

In [ ]:
# Model train hone ke baad ab summary bilkul theek parameters show karegi
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 500, 64)          │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (64, 64)               │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (64, 64)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,019,269 (7.70 MB)

 Trainable params: 673,089 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,346,180 (5.14 MB)

In [ ]:
# Apni marzi ka koi bhi review test karein
test_review = ["The movie was absolutely fantastic and brilliant! I loved every single moment of it."]

# Preprocessing aur Tokenization apply karein
cleaned_test = [text_preprocessing(r) for r in test_review]
test_sequence = tokenizer.texts_to_sequences(cleaned_test)
padded_test = pad_sequences(test_sequence, maxlen=500, padding="post")

# Prediction karein (0 = Negative, 1 = Positive)
prediction = model.predict(padded_test)
print("Prediction Score:", prediction[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
Prediction Score: 0.944756
